# Baseline Quality Analysis - Agent Workflow

**Purpose**: Establish quality baseline for agent workflow (Step 1)
**Focus**: Agent workflow with 2 iterations
**Vocabulary**: 30 French words (A1-C2 levels)

In [ ]:
import json, sys, os, datetime
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'src'))

from dotenv import load_dotenv
load_dotenv()

print('Environment loaded')
print('API key set:', 'MISTRAL_API_KEY' in os.environ)

In [ ]:
from language_learner.config import get_settings
from language_learner.core.llm_client import MistralLLMClient
from language_learner.exercises.generator import ExerciseGenerator
from language_learner.exercises.agents.exercise_reviewer import ExerciseReviewerAgent
from language_learner.models.exercise import Exercise

print('All imports successful')

In [ ]:
VOCAB_FILE = './data/baseline_vocabulary.fr.json'

with open(VOCAB_FILE, 'r', encoding='utf-8') as f:
    vocabulary_words = json.load(f)

print(f'Loaded {len(vocabulary_words)} vocabulary words')
print(f'Sample: {vocabulary_words[:5]}')

In [ ]:
settings = get_settings()
llm = MistralLLMClient(
    model=settings.mistral_model,
    rate_limit=settings.llm_rate_limit,
)
print('LLM client initialized successfully')

In [ ]:
print('Generating exercises with agent workflow (2 iterations)...')
generator = ExerciseGenerator(llm)
exercises = generator.generate_exercises(vocabulary_words[:15])
print(f'Generated {len(exercises)} exercises')

In [ ]:
print('\nAll Generated Exercises:')
print('-' * 80)
for i, e in enumerate(exercises, 1):
    q = e.question[:70] + '...' if len(e.question) > 70 else e.question
    print(f'{i}. [{e.exercise_type.value}] {q}')
    print(f'   Answer: {e.correct_answer}')
    print()

In [ ]:
print('Reviewing all exercises with quality assessment...')
reviewer = ExerciseReviewerAgent(llm)
reviewer_node = reviewer.create_node()

state = {
    'generated_exercises': exercises,
    'reviewed_exercises': [],
    'rejected_exercises': [],
    'feedback': [],
    'iteration': 1
}

result = reviewer_node(state)
approved = result['reviewed_exercises']
rejected = result['rejected_exercises']

print(f'Review complete: {len(approved)} approved, {len(rejected)} rejected')

In [ ]:
total = len(exercises)
approval_rate = (len(approved) / total * 100) if total > 0 else 0

total_score_approved = 0
score_count_approved = 0
total_score_rejected = 0
score_count_rejected = 0
rejection_reasons = {}

for r in rejected:
    if 'quality_score' in r:
        total_score_rejected += r['quality_score']
        score_count_rejected += 1
    reason = r.get('reason', 'unknown')
    rejection_reasons[reason] = rejection_reasons.get(reason, 0) + 1

for a in approved:
    if 'quality_score' in a:
        total_score_approved += a['quality_score']
        score_count_approved += 1

total_score = total_score_approved + total_score_rejected
score_count = score_count_approved + score_count_rejected
avg_score = (total_score / score_count) if score_count > 0 else 0

avg_score_approved = (total_score_approved / score_count_approved) if score_count_approved > 0 else 0
avg_score_rejected = (total_score_rejected / score_count_rejected) if score_count_rejected > 0 else 0

print('=' * 80)
print('QUALITY METRICS')
print('=' * 80)
print(f'Total Exercises: {total}')
print(f'Approved: {len(approved)} ({approval_rate:.2f}%)')
print(f'Rejected: {len(rejected)} ({100-approval_rate:.2f}%)')
print(f'Average Quality Score: {avg_score:.2f}')
print(f'Average Quality Score Approved: {avg_score_approved:.2f}')
print(f'Average Quality Score Rejected: {avg_score_rejected:.2f}')
print(f'Rejection Reasons: {rejection_reasons}')

pass_fail_approval = 'PASS' if approval_rate >= 80 else 'FAIL'
pass_fail_score = 'PASS' if avg_score >= 75 else 'FAIL'
print('\nTARGETS:')
print(f'  Approval Rate: {approval_rate:.2f}% (>80%) - {pass_fail_approval}')
print(f'  Avg Score: {avg_score:.2f} (>75) - {pass_fail_score}')

In [ ]:
print('\n' + '=' * 80)
print('FAILED EXERCISES WITH ASSESSMENTS')
print('=' * 80)

if rejected:
    print(f'Total Failed: {len(rejected)}\n')
    for i, r in enumerate(rejected, 1):
        exercise = r.get('exercise')
        if exercise:
            print(f'{i}. [{exercise.exercise_type.value}]')
            print(f'   Question: {exercise.question}')
            print(f'   Answer: {exercise.correct_answer}')
            print(f'   Context: {exercise.context or "None"}')
            print(f'   Difficulty: {exercise.difficulty.value}')
            print()
            print('   ASSESSMENT:')
            print(f'   Reason: {r.get("reason", "unknown")}')
            print(f'   Feedback: {r.get("feedback", "No feedback")}')
            details = r.get('details', {})
            if details:
                score = details.get('quality_score', 'N/A')
                suggestions = details.get('suggestions', 'None')
                print(f'   Quality Score: {score}')
                print(f'   Suggestions: {suggestions}')
            print()
            print('-' * 80)
else:
    print('All exercises passed quality review!')

In [ ]:
exp_out_dir = Path('/workspaces/language-learner-assistant/experiments/out')
exp_out_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Save approved
approved_data = []
for a in approved:
    e = a['exercise']
    approved_data.append({
        'exercise_id': e.exercise_id,
        'type': e.exercise_type.value,
        'question': e.question,
        'answer': e.correct_answer,
        'context': e.context,
        'difficulty': e.difficulty.value,
        'options': e.options
    })

with open(exp_out_dir / f'approved_{timestamp}.json', 'w', encoding='utf-8') as f:
    json.dump(approved_data, f, indent=2, ensure_ascii=False)

# Save rejected
rejected_data = []
for r in rejected:
    ex = r.get('exercise')
    rejected_data.append({
        'exercise_id': ex.exercise_id if ex else None,
        'type': ex.exercise_type.value if ex else None,
        'question': ex.question if ex else None,
        'answer': ex.correct_answer if ex else None,
        'reason': r.get('reason'),
        'feedback': r.get('feedback'),
        'details': r.get('details')
    })

with open(exp_out_dir / f'rejected_{timestamp}.json', 'w', encoding='utf-8') as f:
    json.dump(rejected_data, f, indent=2, ensure_ascii=False, default=str)

# Save metrics
metrics = {
    'timestamp': timestamp,
    'total': total,
    'approved': len(approved),
    'rejected': len(rejected),
    'approval_rate': approval_rate,
    'avg_score': avg_score,
    'avg_score_approved': avg_score_approved,
    'avg_score_rejected': avg_score_rejected,
    'rejection_reasons': rejection_reasons,
    'target_approval': 80,
    'target_score': 75
}

with open(exp_out_dir / f'metrics_{timestamp}.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print(f'\nResults saved to: {exp_out_dir}/')
print(f'  approved_{timestamp}.json')
print(f'  rejected_{timestamp}.json')
print(f'  metrics_{timestamp}.json')